# Bölüm 7: Sayısal Veriler için Çıkarım
**Haydar Kılıç — Yapay Zeka Mühendisliği**

Bu notebook, **t dağılımı**, **eşleştirilmiş veriler**, **iki ortalamanın farkı** ve **ANOVA** konularını Python ile uygulamalı olarak ele almaktadır.

**Konular:**
1. t Dağılımı — Nedir, Normal'den Farkı
2. Tek Örneklem t-Testi — 13'ü Uğursuz Cuma
3. Eşleştirilmiş Veriler (Paired t-test) — Okuma & Yazma Puanları
4. İki Bağımsız Örneklem t-Testi — Elmas Fiyatları
5. İki Örneklemli Test için Güç Hesaplama
6. ANOVA — Wolf Nehri Aldrin Konsantrasyonu
7. Çoklu Karşılaştırmalar (Bonferroni Düzeltmesi)

In [ ]:
# Gerekli kütüphaneler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from scipy.stats import t, f, norm
import warnings
warnings.filterwarnings('ignore')

# Görsel ayarlar
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
sns.set_style('whitegrid')

print('Kütüphaneler başarıyla yüklendi.')

---
## 1. t Dağılımı

### Neden t dağılımı?

Kitle standart sapması **σ bilinmediğinde** (pratikte neredeyse her zaman böyledir), standart hatayı örneklem standart sapması **s** ile tahmin ederiz:

$$SH = \frac{s}{\sqrt{n}}$$

Ancak **n küçükse** s, σ için güvenilir bir tahmin değildir. Bu belirsizliği hesaba katmak için **t dağılımı** kullanılır.

**Özellikleri:**
- Tıpkı standart normal gibi **0 merkezli** ve simetriktir
- Normal dağılımdan **daha kalın kuyrukları** vardır
- Tek parametresi: **serbestlik derecesi (sd = n − 1)**
- sd → ∞ iken t dağılımı **normale yaklaşır**

In [ ]:
x = np.linspace(-5, 5, 500)
normal_pdf = norm.pdf(x)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Normal vs farklı sd'ler
ax = axes[0]
ax.plot(x, normal_pdf, 'k-', lw=2.5, label='Normal', zorder=5)
colors = ['#e74c3c', '#e67e22', '#3498db', '#95a5a6']
dfs = [1, 2, 5, 10]
for df_val, color in zip(dfs, colors):
    ax.plot(x, t.pdf(x, df=df_val), color=color, lw=1.8,
            linestyle='--', label=f't, sd={df_val}', alpha=0.85)
ax.set_title('t Dağılımı vs Normal Dağılım')
ax.set_xlabel('Değer')
ax.set_ylabel('Yoğunluk')
ax.legend()
ax.set_xlim(-5, 5)
ax.set_ylim(0, 0.45)

# Sağ: Kalın kuyruk vurgusu
ax2 = axes[1]
ax2.plot(x, normal_pdf, 'k-', lw=2.5, label='Normal', zorder=5)
ax2.plot(x, t.pdf(x, df=3), '#e74c3c', lw=2, linestyle='--', label='t, sd=3')
# Kuyrukları renklendir
tail = x[x > 2]
ax2.fill_between(tail, norm.pdf(tail), t.pdf(tail, df=3),
                 alpha=0.4, color='#e74c3c', label='Ekstra kalın kuyruk')
tail_l = x[x < -2]
ax2.fill_between(tail_l, norm.pdf(tail_l), t.pdf(tail_l, df=3),
                 alpha=0.4, color='#e74c3c')
ax2.set_title('t Dağılımının Kalın Kuyrukları (sd=3)')
ax2.set_xlabel('Değer')
ax2.set_ylabel('Yoğunluk')
ax2.legend()
ax2.set_xlim(-5, 5)
ax2.set_ylim(0, 0.45)

plt.tight_layout()
plt.suptitle('Şekil 1: t Dağılımı', fontsize=14, fontweight='bold', y=1.02)
plt.show()

# Sayısal karşılaştırma
print('=== Normal vs t: P(|X| > 2) olasılıkları ===')
print(f'Normal:   {2 * norm.sf(2):.4f}')
for df_val in [1, 2, 5, 10, 30]:
    print(f't(sd={df_val:2d}): {2 * t.sf(2, df=df_val):.4f}')

**Gözlem:** sd arttıkça t dağılımı normale yaklaşır. sd=30 sonrasında fark ihmal edilebilir düzeydedir.

---
## 2. Tek Örneklem t-Testi: 13'ü Uğursuz Cuma

### Araştırma Sorusu
1990–1992 yılları arasında İngiltere'de, 6'sı ve 13'ü Cuma günlerindeki trafik akışı karşılaştırıldı. **İnsanlar 13'ü Cuma'da daha az mı araç kullanıyor?**

### Hipotezler
$$H_0: \mu_{fark} = 0 \quad \text{(6'sı ile 13'ü arasında fark yok)}$$
$$H_A: \mu_{fark} \neq 0 \quad \text{(iki yönlü)}$$

### Test İstatistiği
$$T_{sd} = \frac{\bar{x}_{fark} - 0}{SH}, \quad SH = \frac{s_{fark}}{\sqrt{n}}, \quad sd = n-1$$

In [ ]:
# Veri: 6'sı ve 13'ü Cuma trafik akışları
data_friday = pd.DataFrame({
    'tarih':   ['1990 Tem', '1990 Tem', '1991 Eyl', '1991 Eyl',
                '1991 Ara', '1991 Ara', '1992 Mar', '1992 Mar',
                '1992 Kas', '1992 Kas'],
    'alti':    [139246, 134012, 137055, 133732, 123552, 121139,
                128293, 124631, 124609, 117584],
    'onuc':    [138548, 132908, 136018, 131843, 121641, 118723,
                125532, 120249, 122770, 117263],
    'konum':   ['K1','K2','K1','K2','K1','K2','K1','K2','K1','K2']
})

data_friday['fark'] = data_friday['alti'] - data_friday['onuc']

print('=== 13\'ü Uğursuz Cuma Veri Seti ===')
print(data_friday[['tarih','konum','alti','onuc','fark']].to_string(index=False))

x_bar = data_friday['fark'].mean()
s     = data_friday['fark'].std(ddof=1)
n     = len(data_friday)
se    = s / np.sqrt(n)
sd    = n - 1
T_obs = (x_bar - 0) / se
p_val = 2 * t.sf(abs(T_obs), df=sd)

print(f'\n=== Özet İstatistikler ===')
print(f'Ortalama fark (x̄):  {x_bar:.2f} araç')
print(f'Std sapma (s):      {s:.2f}')
print(f'n:                  {n}')
print(f'Standart hata (SH): {se:.2f}')
print(f'\n=== Hipotez Testi ===')
print(f'T istatistiği:      {T_obs:.3f}')
print(f'Serbestlik derecesi:{sd}')
print(f'p-değeri:           {p_val:.6f}')
print(f'\nKarar (α=0.05):     {"H₀ REDDEDİLİR ✓" if p_val < 0.05 else "H₀ REDDEDİLEMEZ"}')

In [ ]:
# Görselleştirme: Fark histogramı + t dağılımı + p-değeri
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Farkların histogramı
ax1 = axes[0]
ax1.hist(data_friday['fark'], bins=6, color='#3498db', alpha=0.7,
         edgecolor='white', linewidth=0.8)
ax1.axvline(x_bar, color='#e74c3c', lw=2, linestyle='--',
            label=f'x̄ = {x_bar:.0f}')
ax1.set_title('Trafik Akışı Farkları (6\'sı − 13\'ü)')
ax1.set_xlabel('Fark (araç sayısı)')
ax1.set_ylabel('Frekans')
ax1.legend()

# Sağ: t dağılımı üzerinde T istatistiği ve p-değeri
ax2 = axes[1]
x_range = np.linspace(-6, 6, 500)
y_t = t.pdf(x_range, df=sd)
ax2.plot(x_range, y_t, 'k-', lw=2)

# p-değeri bölgelerini gölgele
tail_r = x_range[x_range >= T_obs]
tail_l = x_range[x_range <= -T_obs]
ax2.fill_between(tail_r, t.pdf(tail_r, df=sd), alpha=0.4,
                 color='#e74c3c', label=f'p-değeri = {p_val:.4f}')
ax2.fill_between(tail_l, t.pdf(tail_l, df=sd), alpha=0.4, color='#e74c3c')
ax2.axvline(T_obs,  color='#e74c3c', lw=2, linestyle='--',
            label=f'T = {T_obs:.2f}')
ax2.axvline(-T_obs, color='#e74c3c', lw=2, linestyle='--')
ax2.set_title(f't Dağılımı (sd={sd}) — Hipotez Testi')
ax2.set_xlabel('t değeri')
ax2.set_ylabel('Yoğunluk')
ax2.legend()

plt.tight_layout()
plt.suptitle('Şekil 2: 13\'ü Uğursuz Cuma — Tek Örneklem t-Testi',
             fontsize=13, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# %95 Güven Aralığı
alpha   = 0.05
t_star  = t.ppf(1 - alpha/2, df=sd)  # Kritik t değeri
margin  = t_star * se
ci_low  = x_bar - margin
ci_high = x_bar + margin

print('=== %95 Güven Aralığı ===')
print(f'Kritik t* (sd=9, α/2=0.025): {t_star:.4f}')
print(f'Hata Payı (HP = t* × SH):    {margin:.2f}')
print(f'GA: {x_bar:.0f} ± {margin:.0f}')
print(f'GA: ({ci_low:.0f}, {ci_high:.0f})')
print()
print('Yorum: %95 güvenle 6\'sı Cuma ile 13\'ü Cuma arasındaki')
print(f'ortalama trafik farkının {ci_low:.0f} ile {ci_high:.0f} araç arasında olduğunu söyleyebiliriz.')
print()
print(f'GA sıfırı içeriyor mu? {"Evet" if ci_low <= 0 <= ci_high else "Hayır"}')
print('→ HT ve GA sonuçları tutarlı!')

---
## 3. Eşleştirilmiş Veriler (Paired t-test): Okuma & Yazma Puanları

### Neden Eşleştirilmiş?

Aynı öğrenciler hem okuma hem de yazma testini almıştır. Dolayısıyla iki ölçüm **bağımlıdır** — her öğrenci için bir çift gözlem vardır.

**Strateji:** Farkları al → tek örneklem t-testi uygula

$$\text{fark}_i = \text{okuma}_i - \text{yazma}_i$$

$$H_0: \mu_{fark} = 0, \quad H_A: \mu_{fark} \neq 0$$

In [ ]:
# Gerçekçi veri simüle edelim (sunumdaki istatistiklerle uyumlu)
# x̄_fark = -0.545, s_fark = 8.887, n = 200
np.random.seed(42)
n_students = 200

# Ortalama ve kovaryans matrisi ile çok değişkenli normal
mu    = [51.0, 51.5]
cov   = [[100, 75],   # okuma ve yazma pozitif ilişkili
         [75,  90]]
scores = np.random.multivariate_normal(mu, cov, n_students)
scores = np.clip(np.round(scores), 20, 80).astype(int)

df_scores = pd.DataFrame({'okuma': scores[:,0], 'yazma': scores[:,1]})
df_scores['fark'] = df_scores['okuma'] - df_scores['yazma']

print('İlk 5 satır:')
print(df_scores.head())
print(f'\nOkuma  → Ortalama: {df_scores["okuma"].mean():.2f}, SS: {df_scores["okuma"].std():.2f}')
print(f'Yazma  → Ortalama: {df_scores["yazma"].mean():.2f}, SS: {df_scores["yazma"].std():.2f}')
print(f'Fark   → Ortalama: {df_scores["fark"].mean():.3f}, SS: {df_scores["fark"].std():.3f}')

In [ ]:
# Sunumdaki değerleri kullanarak analizi tekrarlayalım
x_bar_diff  = -0.545
s_diff      =  8.887
n_students  =  200
se_diff     = s_diff / np.sqrt(n_students)
sd_paired   = n_students - 1
T_paired    = x_bar_diff / se_diff
p_paired    = 2 * t.sf(abs(T_paired), df=sd_paired)

print('=== Eşleştirilmiş t-Test: Okuma − Yazma ===')
print(f'Ortalama fark:       {x_bar_diff}')
print(f'Std sapma (fark):    {s_diff}')
print(f'n:                   {n_students}')
print(f'Standart hata (SH):  {se_diff:.4f}')
print(f'T istatistiği:       {T_paired:.4f}')
print(f'Serbestlik derecesi: {sd_paired}')
print(f'p-değeri:            {p_paired:.4f}')
print(f'Karar (α=0.05):      {"H₀ REDDEDİLİR" if p_paired < 0.05 else "H₀ REDDEDİLEMEZ ✓"}')
print()
print('Yorum: Veriler, ortalama okuma ve yazma puanları arasında')
print('anlamlı bir fark olduğuna dair ikna edici kanıt SUNMAMAKTADIR.')

# %95 GA
t_star_p = t.ppf(0.975, df=sd_paired)
ci_p = (x_bar_diff - t_star_p*se_diff, x_bar_diff + t_star_p*se_diff)
print(f'\n%95 GA: ({ci_p[0]:.3f}, {ci_p[1]:.3f})')
print('→ GA sıfırı içeriyor → HT ile tutarlı.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sol: Kutu grafikleri
ax1 = axes[0]
ax1.boxplot([df_scores['okuma'], df_scores['yazma']],
            labels=['Okuma', 'Yazma'],
            patch_artist=True,
            boxprops=dict(facecolor='#3498db', alpha=0.6),
            medianprops=dict(color='black', lw=2))
ax1.set_title('Okuma ve Yazma Puanları')
ax1.set_ylabel('Puan')

# Orta: Farkların histogramı
ax2 = axes[1]
ax2.hist(df_scores['fark'], bins=20, color='#2ecc71', alpha=0.7,
         edgecolor='white')
ax2.axvline(df_scores['fark'].mean(), color='#e74c3c', lw=2,
            linestyle='--', label=f'x̄ = {df_scores["fark"].mean():.2f}')
ax2.axvline(0, color='black', lw=1.5, linestyle=':')
ax2.set_title('Farkların Dağılımı (Okuma − Yazma)')
ax2.set_xlabel('Fark')
ax2.set_ylabel('Frekans')
ax2.legend()

# Sağ: t dağılımı ve p-değeri
ax3 = axes[2]
x_r = np.linspace(-4, 4, 500)
ax3.plot(x_r, t.pdf(x_r, df=sd_paired), 'k-', lw=2)
ax3.axvline(T_paired,  color='#e74c3c', lw=2, linestyle='--',
            label=f'T = {T_paired:.2f}')
ax3.axvline(-T_paired, color='#e74c3c', lw=2, linestyle='--')
# Kuyruklar
t_lo = x_r[x_r <= T_paired]
t_hi = x_r[x_r >= -T_paired]
ax3.fill_between(t_lo, t.pdf(t_lo, df=sd_paired), alpha=0.4, color='#e74c3c')
ax3.fill_between(t_hi, t.pdf(t_hi, df=sd_paired), alpha=0.4, color='#e74c3c',
                 label=f'p = {p_paired:.4f}')
ax3.set_title('Hipotez Testi')
ax3.set_xlabel('t değeri')
ax3.set_ylabel('Yoğunluk')
ax3.legend()

plt.tight_layout()
plt.suptitle('Şekil 3: Eşleştirilmiş t-Testi — Okuma & Yazma',
             fontsize=13, fontweight='bold', y=1.02)
plt.show()

---
## 4. İki Bağımsız Örneklem t-Testi: Elmas Fiyatları

### Araştırma Sorusu
0,99 ve 1 karatlık elmaslar arasında çıplak gözle fark edilemeyen boyut farkına rağmen, **fiyat farkı var mı?**

### Hipotezler (Tek yönlü)
$$H_0: \mu_{pt99} = \mu_{pt100}$$
$$H_A: \mu_{pt99} < \mu_{pt100} \quad (\text{0,99 karatlık daha ucuz})$$

### Test İstatistiği
$$T_{sd} = \frac{(\bar{x}_1 - \bar{x}_2) - 0}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}, \quad sd = \min(n_1-1, n_2-1)$$

In [ ]:
# Sunumdaki özet istatistikler
x_bar_99  = 44.50;  s_99  = 13.32;  n_99  = 23
x_bar_100 = 53.43;  s_100 = 12.22;  n_100 = 30

# Gerçekçi veri simüle et (özet istatistiklere uygun)
np.random.seed(7)
pt99  = np.random.normal(x_bar_99,  s_99,  n_99).round(1)
pt100 = np.random.normal(x_bar_100, s_100, n_100).round(1)

print('0,99 karat:', pt99)
print('\n1 karat:  ', pt100)

# Standart hata ve T istatistiği
se_two  = np.sqrt(s_99**2/n_99 + s_100**2/n_100)
sd_two  = min(n_99 - 1, n_100 - 1)   # = 22
T_two   = (x_bar_99 - x_bar_100) / se_two
p_two   = t.cdf(T_two, df=sd_two)    # Tek yönlü (sol kuyruk)

print('\n=== İki Bağımsız Örneklem t-Testi: Elmas Fiyatları ===')
print(f'         0.99 karat  |  1 karat')
print(f'Ort.:    {x_bar_99:.2f}      |  {x_bar_100:.2f}')
print(f'SS:      {s_99:.2f}      |  {s_100:.2f}')
print(f'n:       {n_99}          |  {n_100}')
print(f'\nSH:                 {se_two:.4f}')
print(f'T istatistiği:      {T_two:.4f}')
print(f'Serbestlik derecesi:{sd_two}')
print(f'p-değeri (tek yönlü):{p_two:.6f}')
print(f'\nKarar (α=0.05):     {"H₀ REDDEDİLİR ✓" if p_two < 0.05 else "H₀ REDDEDİLEMEZ"}')

In [ ]:
# %90 Güven Aralığı (tek yönlü α=0.05 → iki yönlü %90)
t_star_90 = t.ppf(0.95, df=sd_two)   # = 1.717
diff_hat  = x_bar_99 - x_bar_100
ci_90     = (diff_hat - t_star_90*se_two, diff_hat + t_star_90*se_two)

print('=== %90 Güven Aralığı (tek yönlü α=0.05\'e eşdeğer) ===')
print(f'Nokta tahmini:      {diff_hat:.2f}')
print(f'Kritik t* (sd=22):  {t_star_90:.4f}')
print(f'%90 GA:             ({ci_90[0]:.2f}, {ci_90[1]:.2f})')
print()
print('Yorum: 0,99 karatlık elmasın ortalama puan fiyatının')
print(f'1 karatlıktan {abs(ci_90[0]):.2f}$ ile {abs(ci_90[1]):.2f}$ daha düşük olduğundan')
print('%90 güvenle emin olabiliriz.')

# Görsel
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax1 = axes[0]
data_box = [pt99, pt100]
bp = ax1.boxplot(data_box, labels=['0,99 karat', '1 karat'],
                 patch_artist=True, notch=False,
                 boxprops=dict(alpha=0.7),
                 medianprops=dict(color='black', lw=2))
colors_box = ['#3498db', '#e74c3c']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
ax1.scatter([1]*n_99 + [2]*n_100,
            np.concatenate([pt99, pt100]),
            alpha=0.4, color=['#3498db']*n_99 + ['#e74c3c']*n_100,
            zorder=5, s=25)
ax1.set_title('Elmas Puan Fiyatları')
ax1.set_ylabel('Fiyat ($/puan)')

ax2 = axes[1]
x_r = np.linspace(-4.5, 1.5, 500)
ax2.plot(x_r, t.pdf(x_r, df=sd_two), 'k-', lw=2, label=f't(sd={sd_two})')
tail = x_r[x_r <= T_two]
ax2.fill_between(tail, t.pdf(tail, df=sd_two), alpha=0.4, color='#e74c3c',
                 label=f'p = {p_two:.4f}')
ax2.axvline(T_two, color='#e74c3c', lw=2, linestyle='--',
            label=f'T = {T_two:.2f}')
ax2.set_title('Hipotez Testi (Tek Yönlü)')
ax2.set_xlabel('t değeri')
ax2.set_ylabel('Yoğunluk')
ax2.legend()

plt.tight_layout()
plt.suptitle('Şekil 4: İki Bağımsız Örneklem t-Testi — Elmas Fiyatları',
             fontsize=13, fontweight='bold', y=1.02)
plt.show()

---
## 5. Güç Analizi (2 Örneklemli Test)

### Karar Tablosu

| | H₀ Reddedilmez | H₀ Reddedilir |
|---|---|---|
| **H₀ Doğru** | ✓ (1−α) | Tip I Hata (α) |
| **Hₐ Doğru** | Tip II Hata (β) | ✓ **Güç** (1−β) |

### Örnek: Kan Basıncı Klinik Denemesi

- σ = 12 mmHg, her grupta n = 100, hedef etki δ = −3 mmHg
- SH = √(12²/100 + 12²/100) = 1.70

In [ ]:
sigma  = 12
n_each = 100
alpha  = 0.05
delta  = -3   # gerçek etki büyüklüğü

se_bp  = np.sqrt(sigma**2/n_each + sigma**2/n_each)
z_crit = norm.ppf(1 - alpha/2)   # = 1.96

# Ret bölgeleri (iki yönlü)
reject_low  = -z_crit * se_bp    # ≈ -3.332
reject_high =  z_crit * se_bp    # ≈ +3.332

# Güç: H₀ reddedilirken gerçek dağılım delta merkezli
z_power = (reject_low - delta) / se_bp
power   = norm.cdf(z_power)

print('=== Kan Basıncı Klinik Denemesi — Güç Analizi ===')
print(f'σ = {sigma}, n = {n_each}/grup, δ = {delta} mmHg, α = {alpha}')
print(f'Standart hata (SH): {se_bp:.4f}')
print(f'Kritik değer (z*):  ±{z_crit:.4f}')
print(f'Ret bölgesi:        < {reject_low:.3f} veya > {reject_high:.3f}')
print(f'Z (güç hesabı):     {z_power:.4f}')
print(f'Güç (1-β):          {power:.4f} = %{power*100:.1f}')
print(f'Tip II hata (β):    {1-power:.4f} = %{(1-power)*100:.1f}')

In [ ]:
# Güç eğrisi: Farklı örneklem büyüklükleri için güç
n_values  = np.array([10, 20, 30, 50, 75, 100, 150, 200, 300, 500, 750, 1000])
powers_bp = []

for n_val in n_values:
    se_val  = np.sqrt(sigma**2/n_val + sigma**2/n_val)
    rej_low = -z_crit * se_val
    z_p     = (rej_low - delta) / se_val
    powers_bp.append(norm.cdf(z_p))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Güç eğrisi
ax1 = axes[0]
ax1.semilogx(n_values, powers_bp, 'b-o', lw=2, markersize=6)
ax1.axhline(0.80, color='#e74c3c', lw=1.5, linestyle='--', label='%80 Güç hedefi')
ax1.axhline(0.90, color='#2ecc71', lw=1.5, linestyle='--', label='%90 Güç hedefi')
ax1.set_xlabel('Grup başına örneklem büyüklüğü (log)')
ax1.set_ylabel('Güç (1−β)')
ax1.set_title('Güç Eğrisi — Kan Basıncı Denemesi')
ax1.set_ylim(0, 1.05)
ax1.legend()
ax1.grid(True, alpha=0.4)

# Sağ: Null ve alternatif dağılım
ax2 = axes[1]
x_plot = np.linspace(-10, 4, 600)
null_dist = norm.pdf(x_plot, 0, se_bp)
alt_dist  = norm.pdf(x_plot, delta, se_bp)
ax2.plot(x_plot, null_dist, 'b-',  lw=2, label='H₀ dağılımı (δ=0)')
ax2.plot(x_plot, alt_dist,  'g-',  lw=2, label=f'Hₐ dağılımı (δ={delta})')
# Güç bölgesi
power_region = x_plot[x_plot <= reject_low]
ax2.fill_between(power_region, norm.pdf(power_region, delta, se_bp),
                 alpha=0.4, color='green', label=f'Güç = {power:.2f}')
# Tip I bölgesi
alpha_region = x_plot[x_plot <= reject_low]
ax2.fill_between(alpha_region, norm.pdf(alpha_region, 0, se_bp),
                 alpha=0.3, color='blue', label=f'α/2 = {alpha/2}')
ax2.axvline(reject_low,  color='red', lw=1.5, linestyle='--', label=f'Ret sınırı={reject_low:.2f}')
ax2.axvline(reject_high, color='red', lw=1.5, linestyle='--')
ax2.set_xlabel('x̄_tedavi − x̄_kontrol')
ax2.set_ylabel('Yoğunluk')
ax2.set_title('Null ve Alternatif Dağılımlar')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.suptitle('Şekil 5: Güç Analizi', fontsize=13, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# %80 güç için gereken örneklem büyüklüğü
# SH = δ / (z_α/2 + z_β)  →  n = 2σ²(z_α/2 + z_β)² / δ²
target_power = 0.80
z_beta  = norm.ppf(target_power)     # = 0.842
n_req   = 2 * sigma**2 * (z_crit + z_beta)**2 / delta**2

print('=== %80 Güç için Gereken Örneklem Büyüklüğü ===')
print(f'z_α/2 = {z_crit:.3f}')
print(f'z_β   = {z_beta:.3f}')
print(f'δ = {delta}, σ = {sigma}')
print(f'n hesaplama: 2 × {sigma}² × ({z_crit:.3f} + {z_beta:.3f})² / {delta}²')
print(f'n = {n_req:.2f}  →  n ≥ {int(np.ceil(n_req))}')
print('(Sunumdaki sonuçla uyumlu: n ≥ 251)')

---
## 6. ANOVA: Wolf Nehri Aldrin Konsantrasyonu

### Araştırma Sorusu
Üç derinlik seviyesi (dip, orta derinlik, yüzey) arasında ortalama aldrin konsantrasyonu farklı mıdır?

### Hipotezler
$$H_0: \mu_D = \mu_O = \mu_Y$$
$$H_A: \text{En az bir ortalama farklıdır}$$

### ANOVA'nın Mantığı

$$F = \frac{\text{Gruplar arası değişkenlik (GOK)}}{\text{Gruplar içi değişkenlik (HOK)}}$$

F büyükse → gruplar arası fark büyük → H₀'ı reddet

In [ ]:
# Wolf Nehri Aldrin Verisi — Sunumdaki değerlerle uyumlu
np.random.seed(2024)

# Grup istatistikleri: n=10, ortalamalar: 6.04, 5.05, 4.20
bottom    = np.array([3.80, 4.80, 4.90, 5.60, 6.50, 6.00, 7.80, 8.80, 6.50, 5.30])
middepth  = np.array([3.20, 3.80, 4.00, 5.30, 5.50, 6.40, 6.60, 4.90, 5.80, 5.30])
surface   = np.array([3.10, 3.60, 3.80, 4.00, 4.20, 4.60, 4.90, 5.20, 4.30, 3.80])

df_aldrin = pd.DataFrame({
    'aldrin':   np.concatenate([bottom, middepth, surface]),
    'derinlik': ['dip']*10 + ['orta']*10 + ['yüzey']*10
})

print('=== Özet İstatistikler ===')
summary = df_aldrin.groupby('derinlik')['aldrin'].agg(['count','mean','std'])
summary.columns = ['n', 'Ortalama', 'Std Sapma']
summary.loc['Genel'] = [30, df_aldrin['aldrin'].mean(), df_aldrin['aldrin'].std()]
print(summary.round(2))

In [ ]:
# ANOVA — Manuel Hesaplama
groups      = [bottom, middepth, surface]
k           = len(groups)
n_total     = sum(len(g) for g in groups)
grand_mean  = np.concatenate(groups).mean()
group_means = [g.mean() for g in groups]
group_ns    = [len(g) for g in groups]

# Serbestlik dereceleri
sd_G = k - 1
sd_T = n_total - 1
sd_H = sd_T - sd_G

# Kareler toplamları
GKT = sum(n_i * (x_i - grand_mean)**2
          for n_i, x_i in zip(group_ns, group_means))
TKT = sum((x - grand_mean)**2 for x in np.concatenate(groups))
HKT = TKT - GKT

# Ortalama kareler
GOK = GKT / sd_G
HOK = HKT / sd_H

# F istatistiği ve p-değeri
F_obs = GOK / HOK
p_anova = 1 - stats.f.cdf(F_obs, dfn=sd_G, dfd=sd_H)

print('=== ANOVA Tablosu (Manuel Hesaplama) ===')
print(f'{"":<15} {"Sd":>5} {"Kareler Top.":>14} {"Ort. Kare":>12} {"F değeri":>10} {"Pr(>F)":>10}')
print('-'*70)
print(f'{"(Grup) derinlik":<15} {sd_G:>5} {GKT:>14.2f} {GOK:>12.2f} {F_obs:>10.2f} {p_anova:>10.4f}')
print(f'{"(Hata) Artıklar":<15} {sd_H:>5} {HKT:>14.2f} {HOK:>12.2f}')
print(f'{"Toplam":<15} {sd_T:>5} {TKT:>14.2f}')
print()
print(f'Karar (α=0.05): {"H₀ REDDEDİLİR ✓" if p_anova < 0.05 else "H₀ REDDEDİLEMEZ"}')
print('Yorum: En az bir derinlik seviyesindeki ortalama aldrin konsantrasyonu')
print('diğerlerinden anlamlı ölçüde farklıdır.')

In [ ]:
# scipy ile doğrulama
F_scipy, p_scipy = stats.f_oneway(bottom, middepth, surface)
print(f'scipy.stats.f_oneway → F = {F_scipy:.4f}, p = {p_scipy:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sol: Dot plot (sunumdaki görsel)
ax1 = axes[0]
colors_d = {'dip': '#e74c3c', 'orta': '#3498db', 'yüzey': '#2ecc71'}
y_pos = {'dip': 2, 'orta': 1, 'yüzey': 0}
y_labels = ['Yüzey', 'Orta Derinlik', 'Dip']

for group_name, group_data in zip(['dip','orta','yüzey'], groups):
    yp = y_pos[group_name]
    ax1.scatter(group_data, [yp]*len(group_data),
                color=colors_d[group_name], alpha=0.7, s=60, zorder=5)
    ax1.hlines(yp, group_data.min(), group_data.max(),
               color=colors_d[group_name], lw=1.5, alpha=0.5)
    m = group_data.mean()
    ax1.scatter(m, yp, color=colors_d[group_name], s=120,
                marker='D', zorder=6, edgecolor='white', lw=1)

ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(y_labels)
ax1.set_xlabel('Aldrin Konsantrasyonu (ng/L)')
ax1.set_title('Üç Derinlik Seviyesi')

# Orta: Kutu grafikleri + varyans kontrolü
ax2 = axes[1]
bp = ax2.boxplot(groups, labels=['Dip\nsd=1.58', 'Orta\nsd=1.10', 'Yüzey\nsd=0.66'],
                 patch_artist=True,
                 medianprops=dict(color='black', lw=2))
for patch, color in zip(bp['boxes'],
                        ['#e74c3c', '#3498db', '#2ecc71']):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax2.set_ylabel('Aldrin (ng/L)')
ax2.set_title('Kutu Grafikleri (Sabit Varyans Kontrolü)')

# Sağ: F dağılımı ve p-değeri
ax3 = axes[2]
x_f = np.linspace(0.001, 10, 500)
ax3.plot(x_f, stats.f.pdf(x_f, sd_G, sd_H), 'k-', lw=2,
         label=f'F(df₁={sd_G}, df₂={sd_H})')
tail_f = x_f[x_f >= F_obs]
ax3.fill_between(tail_f, stats.f.pdf(tail_f, sd_G, sd_H),
                 alpha=0.4, color='#e74c3c', label=f'p = {p_anova:.4f}')
ax3.axvline(F_obs, color='#e74c3c', lw=2, linestyle='--',
            label=f'F = {F_obs:.2f}')
ax3.set_xlabel('F değeri')
ax3.set_ylabel('Yoğunluk')
ax3.set_title('F Dağılımı — p-Değeri')
ax3.set_xlim(0, 10)
ax3.legend()

plt.tight_layout()
plt.suptitle('Şekil 6: ANOVA — Wolf Nehri Aldrin Konsantrasyonu',
             fontsize=13, fontweight='bold', y=1.02)
plt.show()

---
## 7. Çoklu Karşılaştırmalar — Bonferroni Düzeltmesi

ANOVA H₀'ı reddedince **hangi gruplar** farklı sorusu gelir. Her grup çifti için t-testi yaparsak Tip I hata oranı artar.

### Bonferroni Düzeltmesi

$$\alpha^* = \frac{\alpha}{K}, \quad K = \frac{k(k-1)}{2}$$

3 grup için K = 3 karşılaştırma → α* = 0.05/3 = 0.0167

### Havuzlanmış Standart Hata

$$SH = \sqrt{\frac{HOK}{n_1} + \frac{HOK}{n_2}}, \quad sd = n - k$$

In [ ]:
alpha_bonf = 0.05
K          = k * (k - 1) // 2
alpha_star = alpha_bonf / K

print('=== Çoklu Karşılaştırmalar (Bonferroni) ===')
print(f'Grup sayısı: k = {k}')
print(f'Karşılaştırma sayısı: K = k(k-1)/2 = {K}')
print(f'Düzeltilmiş α* = {alpha_bonf}/{K} = {alpha_star:.4f}')
print()

# Tüm çiftler için t-testi (havuzlanmış SH ile)
group_names = ['dip', 'orta', 'yüzey']
pairs       = [('dip','orta'), ('dip','yüzey'), ('orta','yüzey')]
group_dict  = {'dip': bottom, 'orta': middepth, 'yüzey': surface}
sd_error    = n_total - k  # = 27

print(f'{'Karşılaştırma':<20} {'T':>8} {'p (iki yönlü)':>16} {'α*=0.0167':>12} {'Karar':>15}')
print('-'*80)

results = []
for g1, g2 in pairs:
    d1, d2 = group_dict[g1], group_dict[g2]
    se_pool = np.sqrt(HOK/len(d1) + HOK/len(d2))
    T_ij    = (d1.mean() - d2.mean()) / se_pool
    p_ij    = 2 * t.sf(abs(T_ij), df=sd_error)
    karar   = 'H₀ Reddedilir ✓' if p_ij < alpha_star else 'H₀ Reddedilemez'
    results.append({'pair': f'{g1} vs {g2}', 'T': T_ij, 'p': p_ij, 'karar': karar})
    print(f'{g1:>6} vs {g2:<10} {T_ij:>8.3f} {p_ij:>16.4f} {"< 0.0167" if p_ij < alpha_star else "> 0.0167":>12} {karar:>15}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Karşılaştırma p-değerleri vs Bonferroni eşiği
ax1 = axes[0]
pair_labels = [r['pair'] for r in results]
p_vals_all  = [r['p'] for r in results]
bar_colors  = ['#e74c3c' if p < alpha_star else '#95a5a6' for p in p_vals_all]
bars = ax1.bar(pair_labels, p_vals_all, color=bar_colors, alpha=0.8, edgecolor='white')
ax1.axhline(alpha_star, color='navy', lw=2, linestyle='--',
            label=f'α* = {alpha_star:.4f}')
ax1.axhline(0.05, color='gray', lw=1.5, linestyle=':',
            label='α = 0.05')
ax1.set_ylabel('p-değeri')
ax1.set_title('Bonferroni Düzeltmeli Karşılaştırmalar')
ax1.legend()
for bar, pv in zip(bars, p_vals_all):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{pv:.4f}', ha='center', va='bottom', fontsize=9)

# Sağ: Ortalama farklar ve güven aralıkları
ax2 = axes[1]
t_star_bonf = t.ppf(1 - alpha_star/2, df=sd_error)
pair_names  = [r['pair'] for r in results]
diffs       = []
ci_lo_list  = []
ci_hi_list  = []

for g1, g2 in pairs:
    d1, d2  = group_dict[g1], group_dict[g2]
    diff    = d1.mean() - d2.mean()
    se_pool = np.sqrt(HOK/len(d1) + HOK/len(d2))
    margin  = t_star_bonf * se_pool
    diffs.append(diff)
    ci_lo_list.append(diff - margin)
    ci_hi_list.append(diff + margin)

y_pos_c = range(len(pairs))
for i, (diff, lo, hi, res) in enumerate(
        zip(diffs, ci_lo_list, ci_hi_list, results)):
    color = '#e74c3c' if res['karar'].startswith('H₀ R') and '✓' in res['karar'] \
            else '#95a5a6'
    ax2.plot([lo, hi], [i, i], color=color, lw=3, alpha=0.8)
    ax2.scatter(diff, i, color=color, s=100, zorder=5)

ax2.axvline(0, color='black', lw=1.5, linestyle='--')
ax2.set_yticks(list(y_pos_c))
ax2.set_yticklabels([r['pair'] for r in results])
ax2.set_xlabel('Ortalama Fark')
ax2.set_title(f'Güven Aralıkları (Bonferroni, α*={alpha_star:.4f})')
kirmizi = mpatches.Patch(color='#e74c3c', label='H₀ Reddedilir')
gri     = mpatches.Patch(color='#95a5a6', label='H₀ Reddedilemez')
ax2.legend(handles=[kirmizi, gri])

plt.tight_layout()
plt.suptitle('Şekil 7: Çoklu Karşılaştırmalar — Bonferroni Düzeltmesi',
             fontsize=13, fontweight='bold', y=1.02)
plt.show()

print('\nSonuç: Yalnızca dip & yüzey arasındaki fark istatistiksel olarak anlamlıdır.')
print('Dip & orta ve orta & yüzey arasındaki farklar Bonferroni düzeltmesiyle anlamsız kalır.')

---
## Özet: Hangi Testi Ne Zaman Kullanmalıyım?

| Durum | Test | İstatistik |
|---|---|---|
| 1 grup, σ bilinmiyor | Tek örneklem t-testi | $T = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}$ |
| 2 bağımlı grup (eşleştirilmiş) | Paired t-testi | Farklara tek örneklem t |
| 2 bağımsız grup, σ bilinmiyor | İki örneklem t-testi | $T = \frac{(\bar{x}_1-\bar{x}_2)}{\sqrt{s_1^2/n_1+s_2^2/n_2}}$ |
| ≥3 bağımsız grup | ANOVA (F-testi) | $F = GOK/HOK$ |
| ANOVA sonrası ikili | t-test + Bonferroni | α* = α/K |

In [ ]:
print('Notebook tamamlandı!')
print('Kapsanan konular:')
topics = [
    '1. t Dağılımı — Normal\'den Farkı',
    '2. Tek Örneklem t-Testi (13\'ü Uğursuz Cuma)',
    '3. Güven Aralığı Hesaplama',
    '4. Eşleştirilmiş t-Testi (Okuma & Yazma)',
    '5. İki Bağımsız Örneklem t-Testi (Elmas Fiyatları)',
    '6. Güç Analizi & Örneklem Büyüklüğü',
    '7. ANOVA (Wolf Nehri Aldrin)',
    '8. Çoklu Karşılaştırmalar (Bonferroni)'
]
for topic in topics:
    print(f'  ✓ {topic}')